# AFIP — Autonomous Financial Intelligence Pipeline
### Google Colab Quickstart

This notebook runs the complete AFIP pipeline in Google Colab.
You will need:
- A **Groq API key** (free at https://console.groq.com)
- Your **name and email** for the SEC User-Agent header (required by SEC)

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install prefect>=3.0.0 langgraph>=0.2.0 langchain>=0.3.0 langchain-groq>=0.2.0 \
    langchain-community>=0.3.0 sec-edgar-downloader>=5.0.0 docling>=2.0.0 \
    pydantic>=2.0.0 ratelimit>=2.2.1 tenacity>=8.2.0 loguru>=0.7.0 \
    python-dotenv>=1.0.0 sqlitedict>=2.1.0
print('Dependencies installed!')

## Step 2: Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

## Step 3: Upload the AFIP codebase
Upload the `afip/` folder to your Colab session or clone from your repo.

In [ ]:
import os, sys

# If you uploaded the afip/ folder:
AFIP_PATH = '/content/afip'

# Or if cloning from git:
# !git clone https://your-repo-url/afip.git /content/afip

if AFIP_PATH not in sys.path:
    sys.path.insert(0, AFIP_PATH)

print(f'AFIP path set: {AFIP_PATH}')

## Step 4: Configure Environment

In [ ]:
import os
from pathlib import Path

# ── REQUIRED: Set your credentials ─────────────────────────────────────────
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_KEY_HERE'          # Get at console.groq.com
os.environ['SEC_USER_AGENT'] = 'Your Name your@email.com'  # REQUIRED by SEC

# ── Storage: use Google Drive for persistence ───────────────────────────────
BASE = '/content/drive/MyDrive/afip'
os.environ['STORAGE_DIR']   = f'{BASE}/filings'
os.environ['OUTPUT_DIR']    = f'{BASE}/output'
os.environ['ARCHIVE_DIR']   = f'{BASE}/archive'
os.environ['CHECKPOINT_DB'] = f'{BASE}/checkpoints/pipeline.db'
os.environ['LOG_DIR']       = f'{BASE}/logs'

# ── Pipeline settings ───────────────────────────────────────────────────────
os.environ['LLM_PROVIDER']         = 'groq'
os.environ['LLM_MODEL']            = 'llama-3.2-90b-text-preview'
os.environ['CONFIDENCE_THRESHOLD'] = '0.9'
os.environ['MAX_RETRY_LOOPS']      = '3'

# Create directories
for d in [os.environ['STORAGE_DIR'], os.environ['OUTPUT_DIR'],
          os.environ['ARCHIVE_DIR'], os.environ['LOG_DIR']]:
    Path(d).mkdir(parents=True, exist_ok=True)
Path(os.environ['CHECKPOINT_DB']).parent.mkdir(parents=True, exist_ok=True)

print('Configuration complete!')
print(f'Outputs will be saved to: {BASE}/output')

## Step 5: Validate Configuration

In [ ]:
from config.settings import config
try:
    config.validate()
    print('Configuration valid!')
    print(f'Summary: {config.summary()}')
except EnvironmentError as e:
    print(f'Configuration error: {e}')

## Step 6: Run Pipeline for a Single Ticker

In [ ]:
from src.graph import run_pipeline

TICKER = 'AAPL'  # Change this to any ticker

print(f'Running pipeline for {TICKER}...')
result = run_pipeline(ticker=TICKER, filing_type='10-K')

print(f'\nResults for {TICKER}:')
print(f'  Output path:  {result.get("output_path")}')
print(f'  Verified:     {result.get("is_verified")}')
print(f'  Confidence:   {result.get("confidence_score", 0):.2f}')
print(f'  Retries used: {result.get("retry_count", 0)}')

if result.get('extracted_profile'):
    profile = result['extracted_profile']
    print(f'\nDebt Instruments ({len(profile.debt_instruments)}):')
    for inst in profile.debt_instruments:
        print(f'  - {inst.name}: ${inst.amount:,.0f}M {"(" + str(inst.maturity_year) + ")" if inst.maturity_year else ""}')

## Step 7: View the Output JSON

In [ ]:
import json
from pathlib import Path

output_path = result.get('output_path')
if output_path and Path(output_path).exists():
    data = json.loads(Path(output_path).read_text())
    print(json.dumps(data, indent=2))
else:
    print('No output file found')

## Step 8: Run Batch (Multiple Tickers)

In [ ]:
import time

TICKERS = ['AAPL', 'MSFT', 'GOOGL']  # Add more as needed

results = []
for i, ticker in enumerate(TICKERS, 1):
    print(f'[{i}/{len(TICKERS)}] Processing {ticker}...')
    try:
        r = run_pipeline(ticker=ticker)
        status = 'VERIFIED' if r.get('is_verified') else 'UNVERIFIED'
        conf = r.get('confidence_score', 0)
        print(f'  {status} | confidence: {conf:.2f}')
        results.append({'ticker': ticker, 'status': status, 'confidence': conf})
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'ticker': ticker, 'status': 'ERROR', 'error': str(e)})
    
    if i < len(TICKERS):
        time.sleep(2)  # Be respectful to SEC servers

print('\nBatch complete!')
print(results)

## Step 9: Deploy Scheduled Flow to Prefect Cloud (Optional)

In [ ]:
# Set Prefect Cloud credentials
os.environ['PREFECT_API_KEY'] = 'pnu_YOUR_PREFECT_KEY'

# Login and deploy
!prefect cloud login --key $PREFECT_API_KEY

# Deploy with daily post-market schedule
import subprocess
result = subprocess.run(
    ['python', f'{AFIP_PATH}/prefect_flow.py', 'deploy'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr)